# 05 - Bidirectional ConvLSTM U-Net temporal segmentation

This notebook trains and evaluates a target-aligned bidirectional ConvLSTM U-Net while preserving the baseline preprocessing, official EchoNet TRAIN/VAL/TEST split, target-frame masks, loss, optimizer, checkpointing, metrics, prediction export, and logging conventions from `04_temporal_baseline.ipynb`.


In [ ]:
# Kaggle setup. Skip this cell when the environment already satisfies requirements.txt.
%pip install -q monai opencv-python-headless pandas matplotlib tqdm


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

# Kaggle-compatible project discovery. Override PROJECT_ROOT when needed.
def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])

PROJECT_ROOT = first_existing_path([
    os.environ.get('PROJECT_ROOT'),
    '/kaggle/input/echonet-temporal-xai',
    '/kaggle/working/Echonet_temporal_XAI',
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bidirectional_convlstm_unet import build_bidirectional_convlstm_unet
from src.dataset import EchoNetTemporalDataset, load_temporal_metadata, split_by_echonet_filelist
from src.temporal_train import (
    evaluate_temporal,
    fit_temporal,
    get_temporal_loss,
    plot_temporal_history,
    save_json,
    save_temporal_predictions,
    write_experiment_log,
)
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get('ECHONET_RAW_DIR', PROJECT_ROOT / 'data' / 'raw' / 'EchoNet-Dynamic'))
PROCESSED_DIR = Path(os.environ.get('ECHONET_PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed'))
VIDEOS_DIR = RAW_DIR / 'Videos'
RUN_DIR = Path('/kaggle/working/outputs/runs/bidirectional_convlstm_unet') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'runs' / 'bidirectional_convlstm_unet'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
FIGURES_DIR = RUN_DIR / 'figures'
PREDICTIONS_DIR = FIGURES_DIR / 'predictions'

for directory in [RUN_DIR, CHECKPOINT_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw EchoNet directory: {RAW_DIR}')
print(f'Processed dataset directory: {PROCESSED_DIR}')
print(f'Persistent run directory: {RUN_DIR}')


## Configuration

The temporal context is target-aligned. With the default values below, each sample contains 25 frames and the target ED/ES frame is at sequence position 12.


In [ ]:
RUN_MODE = 'smoke'  # change to 'full' for the complete experiment

NUM_FRAMES_BEFORE = 12
NUM_FRAMES_AFTER = 12
TEMPORAL_STRIDE = 2
TARGET_IDX = NUM_FRAMES_BEFORE
SEQUENCE_LENGTH = NUM_FRAMES_BEFORE + 1 + NUM_FRAMES_AFTER

SMOKE_CONFIG = {
    'run_mode': 'smoke',
    'seed': 42,
    'num_frames_before': NUM_FRAMES_BEFORE,
    'num_frames_after': NUM_FRAMES_AFTER,
    'temporal_stride': TEMPORAL_STRIDE,
    'target_idx': TARGET_IDX,
    'sequence_length': SEQUENCE_LENGTH,
    'image_size': [112, 112],
    'epochs': 1,
    'batch_size': 2,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': 16,
    'max_val_samples': 8,
    'max_test_samples': 8,
    'channels': [16, 32, 64, 128],
}

FULL_CONFIG = {
    'run_mode': 'full',
    'seed': 42,
    'num_frames_before': NUM_FRAMES_BEFORE,
    'num_frames_after': NUM_FRAMES_AFTER,
    'temporal_stride': TEMPORAL_STRIDE,
    'target_idx': TARGET_IDX,
    'sequence_length': SEQUENCE_LENGTH,
    'image_size': [112, 112],
    'epochs': 50,
    'batch_size': 4,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': None,
    'max_val_samples': None,
    'max_test_samples': None,
    'channels': [16, 32, 64, 128],
}

config = SMOKE_CONFIG if RUN_MODE == 'smoke' else FULL_CONFIG
save_json(config, RUN_DIR / 'config.json')
config


## Load processed masks and official EchoNet splits

This reuses the same split helper as the baseline notebook. No random fallback split is used.


In [ ]:
metadata_path = PROCESSED_DIR / 'metadata.csv'
file_list_path = RAW_DIR / 'FileList.csv'
assert metadata_path.exists(), f'Processed metadata not found: {metadata_path}'
assert file_list_path.exists(), f'EchoNet FileList.csv not found: {file_list_path}'
assert VIDEOS_DIR.exists(), f'Videos directory not found: {VIDEOS_DIR}'

samples = load_temporal_metadata(metadata_path)
file_list, _ = load_echonet_tables(RAW_DIR)
train_samples, val_samples, test_samples = split_by_echonet_filelist(samples, file_list)

matched_count = len(train_samples) + len(val_samples) + len(test_samples)
assert matched_count == len(samples), (
    f'{len(samples) - matched_count} samples did not match the official EchoNet split.'
)
assert min(len(train_samples), len(val_samples), len(test_samples)) > 0, 'Every official split must contain samples.'

full_split_counts = {
    'train': len(train_samples),
    'validation': len(val_samples),
    'test': len(test_samples),
}

if config['max_train_samples'] is not None:
    train_samples = train_samples[:config['max_train_samples']]
if config['max_val_samples'] is not None:
    val_samples = val_samples[:config['max_val_samples']]
if config['max_test_samples'] is not None:
    test_samples = test_samples[:config['max_test_samples']]

print(f'Total processed labeled frames: {len(samples):,}')
print(f'Official full split counts: {full_split_counts}')
print(f'Active train samples: {len(train_samples):,}')
print(f'Active validation samples: {len(val_samples):,}')
print(f'Active test samples: {len(test_samples):,}')


## Build datasets and loaders


In [ ]:
dataset_kwargs = {
    'videos_dir': VIDEOS_DIR,
    'num_frames_before': config['num_frames_before'],
    'num_frames_after': config['num_frames_after'],
    'temporal_stride': config['temporal_stride'],
    'image_size': tuple(config['image_size']),
}
train_dataset = EchoNetTemporalDataset(train_samples, augment=True, **dataset_kwargs)
val_dataset = EchoNetTemporalDataset(val_samples, augment=False, **dataset_kwargs)
test_dataset = EchoNetTemporalDataset(test_samples, augment=False, **dataset_kwargs)

loader_kwargs = {
    'batch_size': config['batch_size'],
    'num_workers': config['num_workers'],
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': config['num_workers'] > 0,
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)


## Sanity checks

Run these cells before starting full training.


In [ ]:
sample = train_dataset[0]
assert sample['sequence'].shape == (config['sequence_length'], 1, *config['image_size'])
assert sample['mask'].shape == (1, *config['image_size'])
assert int(sample['target_idx']) == config['target_idx']
assert int(sample['frame_indices'][config['target_idx']]) == int(sample['frame_idx'])
assert len(train_dataset) == len(train_samples)
assert len(val_dataset) == len(val_samples)
assert len(test_dataset) == len(test_samples)

print(f"Single sequence shape: {tuple(sample['sequence'].shape)}")
print(f"Target-frame position: {sample['target_idx']}")
print(f"Target frame index: {sample['frame_idx']}")
print(f"Loaded frame indices: {sample['frame_indices'].tolist()}")
print(f"Mask shape: {tuple(sample['mask'].shape)}")
print(f"Official full split counts: {full_split_counts}")


In [ ]:
sample_batch = next(iter(train_loader))
assert sample_batch['sequence'].ndim == 5
assert sample_batch['sequence'].shape[1] == config['sequence_length']
assert sample_batch['sequence'].shape[2] == 1
assert sample_batch['mask'].shape[1:] == (1, *config['image_size'])
assert torch.all(sample_batch['frame_indices'][:, config['target_idx']] == sample_batch['frame_idx'])

model = build_bidirectional_convlstm_unet(
    in_channels=1,
    out_channels=1,
    channels=tuple(config['channels']),
    num_frames_before=config['num_frames_before'],
    num_frames_after=config['num_frames_after'],
).to(device)
model.eval()
with torch.no_grad():
    sanity_logits = model(sample_batch['sequence'].to(device))
assert sanity_logits.shape == sample_batch['mask'].shape

print(f"Sequence batch shape: {tuple(sample_batch['sequence'].shape)}")
print(f"Mask batch shape: {tuple(sample_batch['mask'].shape)}")
print(f"Model output shape: {tuple(sanity_logits.shape)}")
print(f"Train/val/test active samples: {len(train_dataset):,} / {len(val_dataset):,} / {len(test_dataset):,}")


## Train bidirectional ConvLSTM U-Net

The optimizer and Dice+BCE objective match the baseline defaults. Checkpoints, history, and configuration are written incrementally to persistent Kaggle output storage.


In [ ]:
model = build_bidirectional_convlstm_unet(
    in_channels=1,
    out_channels=1,
    channels=tuple(config['channels']),
    num_frames_before=config['num_frames_before'],
    num_frames_after=config['num_frames_after'],
).to(device)
loss_fn = get_temporal_loss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
)

history = fit_temporal(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=config['epochs'],
    output_dir=RUN_DIR,
)
plot_temporal_history(history, FIGURES_DIR / 'training_curves.png')
display(history.tail())


## Best-checkpoint validation and held-out test evaluation


In [ ]:
best_checkpoint_path = CHECKPOINT_DIR / 'best_model.pt'
best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])

val_metrics = evaluate_temporal(model, val_loader, loss_fn, device)
test_metrics = evaluate_temporal(model, test_loader, loss_fn, device)
save_json(test_metrics, RUN_DIR / 'test_metrics.json')

print('Best-checkpoint validation metrics:', val_metrics)
print('Held-out test metrics:', test_metrics)

prediction_count = save_temporal_predictions(
    model=model,
    loader=test_loader,
    device=device,
    output_dir=PREDICTIONS_DIR,
    max_examples=10,
)
write_experiment_log(
    RUN_DIR / 'experiment_log.md',
    config=config,
    val_metrics=val_metrics,
    test_metrics=test_metrics,
)
print(f'Saved prediction examples: {prediction_count}')


## Verify persistent Kaggle outputs


In [ ]:
required_outputs = [
    CHECKPOINT_DIR / 'best_model.pt',
    CHECKPOINT_DIR / 'final_model.pt',
    FIGURES_DIR / 'training_curves.png',
    RUN_DIR / 'history.csv',
    RUN_DIR / 'config.json',
    RUN_DIR / 'test_metrics.json',
    RUN_DIR / 'experiment_log.md',
]
missing_outputs = [path for path in required_outputs if not path.exists()]
assert not missing_outputs, f'Missing required outputs: {missing_outputs}'

processed_images = list((PROCESSED_DIR / 'images').glob('*.png'))
processed_masks = list((PROCESSED_DIR / 'masks').glob('*.png'))
prediction_files = list(PREDICTIONS_DIR.glob('*.png'))
assert len(processed_images) == len(processed_masks), 'Processed image-mask counts differ.'
assert len(prediction_files) >= min(10, len(test_dataset)), 'Expected qualitative predictions were not saved.'

all_run_files = [path for path in RUN_DIR.rglob('*') if path.is_file()]
print(f'Processed image count: {len(processed_images):,}')
print(f'Processed mask count: {len(processed_masks):,}')
print(f'Prediction figure count: {len(prediction_files):,}')
print(f'Total bidirectional ConvLSTM output files: {len(all_run_files):,}')
print(f'Processed dataset available at: {PROCESSED_DIR.resolve()}')
print(f'Experiment outputs available at: {RUN_DIR.resolve()}')
for path in sorted(all_run_files):
    print(f'  - {path}')
